# Reproducing Unsupervised Object Categorization on ETH-80

This document outlines the steps to reproduce the results from the paper on unsupervised object categorization, as summarized in the provided table.  
<!-- \!\[\[Screenshot 2025-07-21 at 6.30.55 PM.png\]\] -->

## **Overview**

The experiment's goal is to cluster images by representing each as a 5x5 Symmetric Positive-Definite (SPD) covariance matrix. This approach is powerful because covariance matrices can effectively summarize the second-order statistics of feature distributions across an image, capturing not just the features themselves but also their inter-correlations (e.g., how pixel intensity changes with location). These matrices are then clustered using K-Means (KM) and Kernel K-Means (KKM) with different metrics. The primary performance measure is **clustering accuracy**, which quantifies how well the resulting clusters match the ground-truth object categories.

## **Step 1: Data Preparation & Feature Extraction**

### **1\. Dataset**

* **Source**: ETH-80 dataset.  
* **Contents**: 8 object categories, 10 objects/category, 41 images/object. This provides a rich base for testing unsupervised categorization across multiple classes.

### **2\. Image Processing**

* The images must be converted to **grayscale** to extract a single intensity value I per pixel. This is a critical step because the feature vector is defined with a single intensity channel. Using a color image would require a different feature definition to handle the multi-channel data, moving away from the paper's methodology.

### **3\. Data Split**

For each of the 80 objects:

* **Parameter Set (for **σ**)**: Randomly select **21 images**. This set is not for "training" in a supervised sense but is used exclusively to determine a suitable value for the kernel parameter σ in the KKM algorithm.  
* **Evaluation Set**: Use the remaining **20 images** (Total: 80×20=1600 images). This is the dataset on which the clustering algorithms will be run and evaluated.

### **4\. Feature Extraction**

For each image in the evaluation set, compute a single 5x5 covariance descriptor.

1. For each pixel, create a 5D feature vector $f:f=\[x,y,I,∣Ix​∣,∣Iy​∣\]^T$  
   * x,y: Normalized pixel coordinates, providing spatial information.  
   * I: Grayscale pixel intensity, providing photometric information.  
   * ∣Ix​∣,∣Iy​∣: Absolute values of intensity gradients, capturing local texture and edge information.  
2. Calculate the 5x5 covariance matrix C over all N pixels in the image:C=N−11​i=1∑N​(fi​−fˉ)(fi​−fˉ)T  
   This yields one 5×5 SPD matrix (C∈Sym5+​) per image. This matrix is a compact descriptor that summarizes the entire image's feature statistics.

## **Step 2: Implementing Clustering Algorithms**

Implement both K-Means (KM) and Kernel K-Means (KKM).  
**Key Requirement**: The K-Means objective function is non-convex, meaning algorithms can easily get stuck in local minima. To mitigate this sensitivity to initialization, run each algorithm **20 times** with different random starting points. For each set of 20 runs, select the single best result—the one that converged to the minimum sum of squared point-to-centroid distances.

### **Metrics and Methods**

#### **1\. Euclidean**

* **KM**: Vectorize the 5×5 covariance matrices into 25-dimensional vectors and apply standard K-Means. This is a baseline approach that naively treats the matrix as a flat vector, ignoring its inherent geometric structure.  
* **KKM**: Use a Gaussian kernel on the vectorized matrices:k(Ci​,Cj​)=exp(−2σ2∥vec(Ci​)−vec(Cj​)∥22​​)

#### **2\. Log-Euclidean (Top Performer)**

* **Transformation**: Map each SPD matrix C from its natural Riemannian manifold to a flat Euclidean space via the matrix logarithm: L=logm(C). This transformation "un-curves" the space, making standard Euclidean operations meaningful.  
* **KM**: Apply standard K-Means directly on the transformed matrices L. The centroid is the simple arithmetic mean of the L matrices, which corresponds to the true geometric mean on the manifold. This is both computationally efficient and theoretically sound.  
* **KKM**: Use a Gaussian kernel with the Log-Euclidean distance, which is simply the Frobenius norm in the log-domain:k(Ci​,Cj​)=exp(−2σ2∥logm(Ci​)−logm(Cj​)∥F2​​)

#### **3\. Power-Euclidean (α=0.5)**

* **KM**: Use the distance d(Ci​,Cj​)=∥Ci0.5​−Cj0.5​∥F​. Centroids must be computed via the **Karcher mean**, an iterative optimization process that finds the intrinsic mean on the manifold. This is computationally expensive.  
* **KKM**: Use a Gaussian kernel based on the Power-Euclidean distance.

#### **4\. Cholesky**

* **KM**: Use a distance based on the Cholesky decomposition. This also requires the computationally intensive **Karcher mean** for centroid computation.  
* **KKM**: Use a Gaussian kernel based on the Cholesky distance.

## **Step 3: Experiment Execution and Evaluation**

1. **Set Number of Classes (**k**)**: Run experiments for k=3,4,5,6,7,8. For k\<8, you need to select a subset of the 8 categories.  
2. **Set Kernel Parameter** σ: Use the "parameter set" (21 images per object) to determine σ. A common and robust heuristic is to set σ as the median of all pairwise distances between these samples. This prevents outliers from skewing the kernel width.  
3. **Evaluate Performance**:  
   * After clustering, you have cluster assignments but no labels. To measure accuracy, you must find the optimal mapping between your algorithm's cluster labels and the ground-truth class labels. This is a classic assignment problem, solved optimally using the **Hungarian algorithm**.  
   * **Clustering Accuracy** is then calculated as the percentage of images assigned to the correct cluster based on this optimal mapping.

### **⚠️ Ambiguities & Challenges**

* **Class Subsets**: The paper doesn't state *which* classes were used for experiments with k\<8. This is a significant source of variability. To ensure robust results, one might run tests on multiple random subsets of classes and report the average accuracy.  
* **Karcher Mean**: Implementing the Karcher mean for the Power-Euclidean and Cholesky metrics is complex and computationally demanding. The superior performance and simplicity of the Log-Euclidean metric highlight its significant practical advantages.

In [6]:
import os
import requests
import zipfile

# ETH-80 GitHub repo URL
repo_url = "https://github.com/chenchkx/ETH-80.git"
dataset_dir = "ETH-80"

def clone_eth80(repo_url, dataset_dir):
    if not os.path.exists(dataset_dir):
        print("Cloning ETH-80 dataset from GitHub...")
        os.system(f"git clone {repo_url}")
        print("ETH-80 dataset cloned.")
    else:
        print("ETH-80 dataset already exists.")

clone_eth80(repo_url, dataset_dir)

ETH-80 dataset already exists.


In [7]:
# !pip install -q opencv-python
import numpy as np 
import cv2 as cv